In [25]:
import os
import re
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from pydub import AudioSegment
from scipy.fft import fft, fftfreq

# ---------------------------------
# 1. CNN Model Definition (Unchanged)
# ---------------------------------
class CNNModel(nn.Module):
    def __init__(self, in_channels=1):
        super(CNNModel, self).__init__()
        
        # Block 1
        self.conv1 = nn.Conv2d(in_channels=1, 
                               out_channels=1, 
                               kernel_size=(1, 2), 
                               stride=(1, 1), 
                               padding=(0, 0))
        self.relu1 = nn.ReLU()
        
        self.conv2 = nn.Conv2d(in_channels=1, 
                               out_channels=2, 
                               kernel_size=(1, 2), 
                               stride=(1, 1), 
                               padding=(0, 0))
        self.relu2 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=(1, 2), stride=(1, 2))
        
        # Block 2
        self.conv3 = nn.Conv2d(in_channels=2, 
                               out_channels=2, 
                               kernel_size=(1, 2), 
                               stride=(1, 1), 
                               padding=(0, 0))
        self.relu3 = nn.ReLU()
        self.conv4 = nn.Conv2d(in_channels=2, 
                               out_channels=4, 
                               kernel_size=(1, 2), 
                               stride=(1, 1), 
                               padding=(0, 0))
        self.relu4 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=(1, 2), stride=(1, 2))
        
        # Block 3
        self.conv5 = nn.Conv2d(in_channels=4, 
                               out_channels=4, 
                               kernel_size=(1, 2), 
                               stride=(1, 1), 
                               padding=(0, 0))
        self.relu5 = nn.ReLU()
        self.conv6 = nn.Conv2d(in_channels=4, 
                               out_channels=8, 
                               kernel_size=(1, 2), 
                               stride=(1, 1), 
                               padding=(0, 0))
        self.relu6 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(kernel_size=(1, 2), stride=(1, 2))

        # Block 4
        self.conv7 = nn.Conv2d(in_channels=8, 
                               out_channels=8, 
                               kernel_size=(1, 2), 
                               stride=(1, 1), 
                               padding=(0, 0))
        self.relu7 = nn.ReLU()
        self.conv8 = nn.Conv2d(in_channels=8, 
                               out_channels=16, 
                               kernel_size=(1, 2), 
                               stride=(1, 1), 
                               padding=(0, 0))
        self.relu8 = nn.ReLU()
        self.pool4 = nn.MaxPool2d(kernel_size=(1, 2), stride=(1, 2))

        # Block 5
        self.conv9 = nn.Conv2d(in_channels=16, 
                               out_channels=16, 
                               kernel_size=(1, 2), 
                               stride=(1, 1), 
                               padding=(0, 0))
        self.relu9 = nn.ReLU()
        self.conv10 = nn.Conv2d(in_channels=16, 
                                out_channels=16, 
                                kernel_size=(1, 2), 
                                stride=(1, 1), 
                                padding=(0, 0))
        self.relu10 = nn.ReLU()
        self.pool5 = nn.MaxPool2d(kernel_size=(1, 2), stride=(1, 2))

        # Block 6
        self.conv11 = nn.Conv2d(in_channels=16, 
                                out_channels=16, 
                                kernel_size=(1, 2), 
                                stride=(1, 1), 
                                padding=(0, 0))
        self.relu11 = nn.ReLU()
        self.conv12 = nn.Conv2d(in_channels=16, 
                                out_channels=32, 
                                kernel_size=(1, 2), 
                                stride=(1, 1), 
                                padding=(0, 0))
        self.relu12 = nn.ReLU()
        self.pool6 = nn.MaxPool2d(kernel_size=(1, 2), stride=(1, 2))

        # Block 7
        self.conv13 = nn.Conv2d(in_channels=32, 
                                out_channels=32, 
                                kernel_size=(1, 2), 
                                stride=(1, 1), 
                                padding=(0, 0))
        self.relu13 = nn.ReLU()
        self.conv14 = nn.Conv2d(in_channels=32, 
                                out_channels=32, 
                                kernel_size=(1, 2), 
                                stride=(1, 1), 
                                padding=(0, 0))
        self.relu14 = nn.ReLU()
        self.pool7 = nn.MaxPool2d(kernel_size=(1, 2), stride=(1, 2))

        # Block 8
        self.conv15 = nn.Conv2d(in_channels=32, 
                                out_channels=32, 
                                kernel_size=(1, 2), 
                                stride=(1, 1), 
                                padding=(0, 0))
        self.relu15 = nn.ReLU()
        self.conv16 = nn.Conv2d(in_channels=32, 
                                out_channels=64, 
                                kernel_size=(1, 2), 
                                stride=(1, 1), 
                                padding=(0, 0))
        self.relu16 = nn.ReLU()
        self.pool8 = nn.MaxPool2d(kernel_size=(1, 2), stride=(1, 2))

        # Block 9
        self.conv17 = nn.Conv2d(in_channels=64, 
                                out_channels=64, 
                                kernel_size=(1, 2), 
                                stride=(1, 1), 
                                padding=(0, 0))
        self.relu17 = nn.ReLU()
        self.conv18 = nn.Conv2d(in_channels=64, 
                                out_channels=64, 
                                kernel_size=(1, 2), 
                                stride=(1, 1), 
                                padding=(0, 0))
        self.relu18 = nn.ReLU()
        self.conv19 = nn.Conv2d(in_channels=64, 
                                out_channels=128, 
                                kernel_size=(1, 2), 
                                stride=(1, 1), 
                                padding=(0, 0))
        self.relu19 = nn.ReLU()
        self.pool9 = nn.MaxPool2d(kernel_size=(1, 2), stride=(1, 2))

        # Block 10
        self.conv20 = nn.Conv2d(in_channels=128, 
                                out_channels=128, 
                                kernel_size=(1, 2), 
                                stride=(1, 1), 
                                padding=(0, 0))
        self.relu20 = nn.ReLU()
        self.conv21 = nn.Conv2d(in_channels=128, 
                                out_channels=128, 
                                kernel_size=(1, 2), 
                                stride=(1, 1), 
                                padding=(0, 0))
        self.relu21 = nn.ReLU()
        self.conv22 = nn.Conv2d(in_channels=128, 
                                out_channels=128, 
                                kernel_size=(1, 2), 
                                stride=(1, 1), 
                                padding=(0, 0))
        self.relu22 = nn.ReLU()
        self.conv23 = nn.Conv2d(in_channels=128, 
                                out_channels=256, 
                                kernel_size=(1, 2), 
                                stride=(1, 1), 
                                padding=(0, 0))
        self.relu23 = nn.ReLU()
        self.pool10 = nn.MaxPool2d(kernel_size=(1, 2), stride=(1, 2))

        self.flatten = nn.Flatten()
        # Adjust input dimension if needed
        self.fc1 = nn.Linear(12288, 2048)
        self.fc2 = nn.Linear(2048, 256)
        self.fc3 = nn.Linear(256, 8)
        self.s = nn.Softmax(dim=1)

    def forward(self, x):
        # Block 1
        x = self.pool1(self.relu2(self.conv2(self.relu1(self.conv1(x)))))
        # Block 2
        x = self.pool2(self.relu4(self.conv4(self.relu3(self.conv3(x)))))
        # Block 3
        x = self.pool3(self.relu6(self.conv6(self.relu5(self.conv5(x)))))
        # Block 4
        x = self.pool4(self.relu8(self.conv8(self.relu7(self.conv7(x)))))
        # Block 5
        x = self.pool5(self.relu10(self.conv10(self.relu9(self.conv9(x)))))
        # Block 6
        x = self.pool6(self.relu12(self.conv12(self.relu11(self.conv11(x)))))
        # Block 7
        x = self.pool7(self.relu14(self.conv14(self.relu13(self.conv13(x)))))
        # Block 8
        x = self.pool8(self.relu16(self.conv16(self.relu15(self.conv15(x)))))
        # Block 9
        x = self.pool9(self.relu19(self.conv19(self.relu18(self.conv18(self.relu17(self.conv17(x)))))))
        # Block 10
        x = self.pool10(self.relu23(self.conv23(self.relu22(
            self.conv22(self.relu21(
                self.conv21(self.relu20(
                    self.conv20(x)
                ))
            ))
        ))))

        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc3(x)
        x = self.s(x)
        return x

In [26]:
import os
import re
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from pydub import AudioSegment
from scipy.fft import fft, fftfreq

def transform_raw_wav(file_path, target_sr=16000):
    """
    Load a WAV file, convert to a target sample rate (SR) of 16,000 Hz,
    convert to mono, and return the samples as a NumPy array.
    """
    audio = AudioSegment.from_wav(file_path)
    audio = audio.set_frame_rate(target_sr)
    audio = audio.set_channels(1)
    audio_data_bytes = np.array(audio.get_array_of_samples(), dtype=np.int16).tobytes()
    samples = np.frombuffer(audio_data_bytes, dtype=np.int16)
    return samples

def fouriertransform(y, sample_rate=16000):
    """
    Perform an FFT on a NumPy array of audio samples at 16 kHz
    and return [freqs, amplitudes]. Only the positive half is returned.
    """
    N = len(y)
    T = 1.0 / sample_rate
    yf = fft(y)
    xf = fftfreq(N, T)
    # Keep positive frequencies
    return [xf[:N//2], 2.0/N * np.abs(yf[:N//2])]

def load_dataset_and_labels(dataset_dir="dataset"):
    """
    Loads all WAV files from each of the 8 subfolders (directions).
    Each folder has files named like: device_i_j_something.wav
      - i in {1,2,3,4}
      - j is the "set index"
    We want to group by j so that device_1_j, device_2_j, device_3_j, device_4_j
    get stacked into a shape (4, 16000).

    Returns:
        - X: list of stacked Fourier amplitude matrices, shape (4, 16000) each
        - Y: list of integer labels for each matrix (0..7 if folder names are 1..8)
    """
    X = []
    Y = []

    # We expect subfolders named '1', '2', ..., '8'
    direction_folders = [str(i) for i in range(1, 9)]
    
    for direction_str in direction_folders:
        folder_path = os.path.join(dataset_dir, direction_str)
        if not os.path.exists(folder_path):
            continue

        # grouped[j] = dict { 1: amp1, 2: amp2, 3: amp3, 4: amp4 }
        grouped = {}

        for fname in os.listdir(folder_path):
            if not fname.endswith(".wav"):
                continue
            
            # Example filename pattern: device_1_3_something.wav
            match = re.match(r"device_(\d)_(\d+)_", fname)
            if match:
                i = int(match.group(1))   # microphone/device index: 1..4
                j = match.group(2)       # set index
                full_path = os.path.join(folder_path, fname)

                # 1) Load the audio at 16k, convert to mono
                samples = transform_raw_wav(full_path, target_sr=16000)
                
                # 2) Perform Fourier Transform
                freq, amp = fouriertransform(samples, sample_rate=16000)
                
                # We only keep the first 16,000 amplitudes
                amp_16k = amp[:16000]

                # Save into dictionary
                if j not in grouped:
                    grouped[j] = {}
                grouped[j][i] = amp_16k
        
        # Now gather only those sets that have devices 1..4
        for j_key, device_dict in grouped.items():
            if all(k in device_dict for k in [1,2,3,4]):
                # Stack them: shape (4, 16000)
                mat = np.stack([
                    device_dict[1],
                    device_dict[2],
                    device_dict[3],
                    device_dict[4],
                ], axis=0)
                
                X.append(mat)
                # Convert direction_str (1..8) to label (0..7)
                label = int(direction_str) - 1
                Y.append(label)

    return X, Y

In [27]:
if __name__ == "__main__":
    # Instantiate model
    model = CNNModel(in_channels=1)

    # Define loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Load all data (X) and labels (Y)
    X, Y = load_dataset_and_labels(dataset_dir="dataset")

    # Convert lists to NumPy arrays / Tensors
    X_np = np.array(X)  # shape: (N, 4, 16000)
    Y_np = np.array(Y)  # shape: (N,)

    print(f"Total examples: {X_np.shape[0]}")
    print(f"X_np shape: {X_np.shape}")  # (N, 4, 16000)
    print(f"Y_np shape: {Y_np.shape}")  # (N,)

    # Reshape X to (N, 1, 4, 16000)+
    X_tensor = torch.tensor(X_np, dtype=torch.float32).unsqueeze(1)  # (N,1,4,16000)
    Y_tensor = torch.tensor(Y_np, dtype=torch.long)                  # (N,)

    # Hyperparameters
    epochs = 10
    batch_size = 13

    # Training Loop
    model.train()  # set model to training mode
    
    N = len(X_tensor)
    for epoch in range(epochs):
        for start_idx in range(0, N, batch_size):
            end_idx = start_idx + batch_size
            if end_idx > N:
                break
            
            Xbatch = X_tensor[start_idx:end_idx]  # shape (B,1,4,16000)
            ybatch = Y_tensor[start_idx:end_idx]  # shape (B,)

            # Forward pass
            y_pred = model(Xbatch)  # shape (B,8)

            # Compute loss
            loss = criterion(y_pred, ybatch)
            
            # Backprop and optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            print(f"Epoch [{epoch+1}/{epochs}], Step [{start_idx+1}/{N}], Loss: {loss.item():.4f}")

    print("Training finished!")

Total examples: 247
X_np shape: (247, 4, 16000)
Y_np shape: (247,)
Epoch [1/10], Step [1/247], Loss: 2.0820
Epoch [1/10], Step [14/247], Loss: 1.4842
Epoch [1/10], Step [27/247], Loss: 1.8894
Epoch [1/10], Step [40/247], Loss: 2.2740
Epoch [1/10], Step [53/247], Loss: 2.2740
Epoch [1/10], Step [66/247], Loss: 2.2740
Epoch [1/10], Step [79/247], Loss: 2.2740
Epoch [1/10], Step [92/247], Loss: 2.2740
Epoch [1/10], Step [105/247], Loss: 2.2740
Epoch [1/10], Step [118/247], Loss: 2.2740
Epoch [1/10], Step [131/247], Loss: 2.2740
Epoch [1/10], Step [144/247], Loss: 2.2740
Epoch [1/10], Step [157/247], Loss: 2.2740
Epoch [1/10], Step [170/247], Loss: 2.2740
Epoch [1/10], Step [183/247], Loss: 2.2740
Epoch [1/10], Step [196/247], Loss: 2.2740
Epoch [1/10], Step [209/247], Loss: 2.2740
Epoch [1/10], Step [222/247], Loss: 2.2740
Epoch [1/10], Step [235/247], Loss: 2.2740
Epoch [2/10], Step [1/247], Loss: 1.2740


KeyboardInterrupt: 

In [23]:
def predict_single_probabilities(model, device_files):
    """
    Given a list of four WAV files (device_1, device_2, device_3, device_4),
    load them, transform them, pass them through the model, and return the 
    probability distribution over all 8 classes.

    Args:
      model: Trained CNN model (in eval mode, with final softmax).
      device_files: List/tuple of 4 file paths in the correct order.
    
    Returns:
      probabilities: A list (or NumPy array) of length 8, 
                     containing the predicted probabilities 
                     for each of the 8 classes.
    """
    assert len(device_files) == 4, "Must provide exactly four .wav files (device_1..device_4)."
    
    # 1) Load and transform each device file
    stacked = []
    for fp in device_files:
        samples = transform_raw_wav(fp, target_sr=16000)
        _, amp = fouriertransform(samples, sample_rate=16000)
        amp_16k = amp[:16000]  # keep first 16,000 amplitude bins
        stacked.append(amp_16k)
    
    # 2) Stack into shape (4, 16000)
    matrix = np.stack(stacked, axis=0)
    
    # 3) Convert to PyTorch tensor of shape (1, 1, 4, 16000)
    matrix_tensor = torch.tensor(matrix, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    
    # 4) Model inference (softmax output → (1,8))
    with torch.no_grad():
        outputs = model(matrix_tensor)  # shape: (1,8)
        # Because your model has self.s(x) at the end, 'outputs' is already a distribution.
        
        # Convert to a 1D array: shape (8,)
        probs = outputs.squeeze().cpu().numpy()
    
    return probs


if __name__ == "__main__":
    # Example usage:
    
    # 1) Load your trained model and set it to eval mode
    # model = CNNModel(in_channels=1)
    # model.load_state_dict(torch.load("model_weights.pt"))
    model.eval()
    
    # 2) Suppose you have these four files from the same set (j):
    device_1_file = "dataset/4/device_1_16_20241106_185835_541427.wav"
    device_2_file = "dataset/4/device_2_16_20241106_185835_540427.wav"
    device_3_file = "dataset/4/device_3_16_20241106_185835_541427.wav"
    device_4_file = "dataset/4/device_4_16_20241106_185835_556564.wav"
    device_files = [device_1_file, device_2_file, device_3_file, device_4_file]

    # 3) Run inference for a single example
    probabilities = predict_single_probabilities(model, device_files)
    
    # 4) Print out the probabilities
    print("Probabilities for each of the 8 classes:")
    for class_index, prob in enumerate(probabilities):
        print(f"Class {class_index}: {prob:.4f}")
    
    # 5) If you also want the predicted class:
    predicted_class = np.argmax(probabilities)
    print(f"\nPredicted Class Index: {predicted_class}")


Probabilities for each of the 8 classes:
Class 0: 1.0000
Class 1: 0.0000
Class 2: 0.0000
Class 3: 0.0000
Class 4: 0.0000
Class 5: 0.0000
Class 6: 0.0000
Class 7: 0.0000

Predicted Class Index: 0
